# **DeceptronNet v0**

### Overview
This notebook implements **DeceptronNet v0**, the single-scale baseline for local inverse learning.  
It provides a clean, fair comparison against classical optimization methods:
**Levenberg–Marquardt (LM)** and **Projected Gradient (X-GD)**, under identical initialization
and stopping criteria.  

The notebook performs:
- Synthetic dataset generation for forward and inverse imaging tasks  
- Training of the DeceptronNet v0 unrolled architecture  
- Evaluation under consistent residual-based stopping rules  
- Quantitative comparison (final RMSE, mean iterations)

UPDATE: L-BFGS and real-world data evaluation is added in seperate notebook, to address reviewer's suggestions


In [1]:
from __future__ import annotations
import os, json, math, time, random, argparse, csv
from dataclasses import dataclass
from typing import Callable, List

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
if not os.path.isdir("outputs"):
    os.mkdir("outputs")
@dataclass
class CFG:
    out_dir: str = "outputs"
    seed: int = 0
    device: str = "auto"     # "cpu", "cuda", or "auto"
    dtype: torch.dtype = torch.float32

    H: int = 64
    W: int = 64
    ds: int = 2

    Ntr: int = 4000
    Nva: int = 400
    Nte: int = 400

    sigma_min: float = 0.7
    sigma_max: float = 2.0
    satur_gamma: float = 1.2
    shot_scale: float = 0.03

    sigma_nom: float = 1.3   # nominal PSF for nominal model

    batch_size: int = 64
    epochs: int = 12
    lr: float = 2e-3
    steps_unroll: int = 6

    eps_y_rel: float = 0.3
    max_iters: int = 80

    ls_alpha: float = 0.5
    ls_beta: float = 0.5
    lm_damp_init: float = 1e-2
    cg_iters: int = 10


def get_device(dev: str) -> torch.device:
    if dev == "auto":
        dev = "cuda" if torch.cuda.is_available() else "cpu"
    return torch.device(dev)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


In [2]:
# --- Core Operators (Gaussian, convolution, sensor, sampling)

def gaussian_kernel(sigma: torch.Tensor, size: int=11, dtype=torch.float32) -> torch.Tensor:
    r = size//2
    xs = torch.arange(-r, r+1, device=sigma.device, dtype=dtype)
    g1 = torch.exp(-0.5*(xs/sigma)**2); g1 = g1/g1.sum()
    k = torch.outer(g1,g1); k = k/k.sum(); return k

def conv2(x: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    B,C,H,W = x.shape
    if k.dim()==2:
        kHW = k.view(1,1,k.shape[0],k.shape[1]).to(x.device, x.dtype)
        return F.conv2d(x, kHW, padding=k.shape[0]//2)
    elif k.dim()==3 and k.shape[0]==B:
        k = k.unsqueeze(1)
    if k.dim()==4 and k.shape[0]==B and k.shape[1]==1:
        x1 = x.permute(1,0,2,3).contiguous()
        y1 = F.conv2d(x1, k.to(x.device, x.dtype), padding=k.shape[-1]//2, groups=B)
        return y1.permute(1,0,2,3).contiguous()
    raise RuntimeError(f"Bad kernel shape {tuple(k.shape)} for input {tuple(x.shape)}")

def downsample(x: torch.Tensor, s: int) -> torch.Tensor:
    return F.avg_pool2d(x, kernel_size=s, stride=s)

def upsample_nn(y: torch.Tensor, s: int) -> torch.Tensor:
    return F.interpolate(y, scale_factor=s, mode="nearest")

def sensor_nl(z: torch.Tensor, gamma: float) -> torch.Tensor:
    return torch.sigmoid(gamma*(z-0.5))

def A_true(cfg: CFG, x: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    y_lin = conv2(x, k).clamp(0,1)
    y_nl  = sensor_nl(y_lin, cfg.satur_gamma)
    return downsample(y_nl, cfg.ds)

class NominalForward:
    def __init__(self, cfg: CFG, device, dtype):
        self.cfg=cfg
        self.k = gaussian_kernel(torch.tensor(cfg.sigma_nom, device=device, dtype=dtype), dtype=dtype)
    def A(self, x: torch.Tensor) -> torch.Tensor:
        return downsample(conv2(x, self.k), self.cfg.ds)

# --- Synthetic dataset generation
@torch.no_grad()
def synth_x(cfg: CFG, B:int, device, dtype, band:float) -> torch.Tensor:
    z = torch.randn(B,1,cfg.H,cfg.W, device=device, dtype=dtype)
    k = gaussian_kernel(torch.tensor(1.0 + 2.0*(1-band), device=device, dtype=dtype), dtype=dtype)
    z = conv2(z, k)
    z = (z - z.amin(dim=(1,2,3), keepdim=True)) / (z.amax(dim=(1,2,3), keepdim=True) - z.amin(dim=(1,2,3), keepdim=True) + 1e-8)
    return z

@torch.no_grad()
def make_dataset(cfg: CFG, device, dtype):
    def gen(N:int):
        Xs, Ys, Ks = [], [], []
        for _ in range(N):
            x = synth_x(cfg, 1, device, dtype, band=np.random.uniform(0.3,0.8))
            sigma = torch.tensor(np.random.uniform(cfg.sigma_min, cfg.sigma_max), device=device, dtype=dtype)
            k = gaussian_kernel(sigma, dtype=dtype)
            y = A_true(cfg, x, k)
            std = cfg.shot_scale * torch.clamp(y,0,1).sqrt()
            y = (y + torch.randn_like(y)*std).clamp(0,1)
            Xs.append(x); Ys.append(y); Ks.append(k.unsqueeze(0))
        return torch.cat(Xs,0), torch.cat(Ys,0), torch.cat(Ks,0)
    return gen(cfg.Ntr), gen(cfg.Nva), gen(cfg.Nte)


In [3]:
# --- DeceptronNet (DNet) model

class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, c_in=3, c_mid=32):
        super().__init__()
        self.e1 = ConvBlock(c_in, c_mid)
        self.p1 = nn.MaxPool2d(2)
        self.e2 = ConvBlock(c_mid, c_mid*2)
        self.u1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.d1 = ConvBlock(c_mid*2 + c_mid, c_mid)
        self.out = nn.Conv2d(c_mid, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.p1(e1))
        u  = self.u1(e2)
        d1 = self.d1(torch.cat([u, e1], dim=1))
        return self.out(d1)

class DNet(nn.Module):
    """DeceptronNet v0 (single-scale, learned step per unroll)"""
    def __init__(self, steps=6):
        super().__init__()
        self.steps = steps
        self.gain  = nn.Parameter(torch.ones(steps)*0.7)
        self.net   = UNetSmall(c_in=3, c_mid=32)
        self.nom   = None
    def set_nominal(self, nominal: NominalForward):
        self.nom = nominal
    def forward_steps(self, y: torch.Tensor, cfg: CFG) -> List[torch.Tensor]:
        assert self.nom is not None, "Call set_nominal(nominal) before forward()"
        x = upsample_nn(y, cfg.ds).clamp(0,1)
        traj = [x.clone()]
        for t in range(self.steps):
            y_pred = self.nom.A(x)
            r = y_pred - y
            feat = torch.cat([upsample_nn(y,cfg.ds), upsample_nn(r,cfg.ds), x], dim=1)
            dx = self.net(feat)
            x = (x - self.gain[t].sigmoid()*dx).clamp(0,1)
            traj.append(x.clone())
        return traj
    def forward(self, y: torch.Tensor, cfg: CFG) -> torch.Tensor:
        return self.forward_steps(y, cfg)[-1]


In [4]:
def train(cfg: CFG, model: DNet, TR, VA, device, dtype, nominal: NominalForward):
    Xtr, Ytr, Ktr = TR
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    iters_per_epoch = int(math.ceil(len(Xtr)/cfg.batch_size))
    model.train()
    for ep in range(cfg.epochs):
        perm = torch.randperm(Xtr.shape[0], device=device)
        ep_loss = 0.0
        for t in range(iters_per_epoch):
            idx = perm[t*cfg.batch_size:(t+1)*cfg.batch_size]
            x_true = Xtr[idx].to(device, dtype)
            y      = Ytr[idx].to(device, dtype)
            k      = Ktr[idx].to(device, dtype)

            model.set_nominal(nominal)
            x_hat = model(y, cfg)
            L_sup = F.mse_loss(x_hat, x_true)
            with torch.no_grad():
                y_cons = A_true(cfg, x_hat, k)
            L_y = F.mse_loss(y_cons, y)
            loss = L_sup + L_y

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += float(loss.item())
        print(f"[ep {ep+1:02d}/{cfg.epochs}] loss={ep_loss/iters_per_epoch:.5f}")


In [5]:
def hvp(grad_fn: Callable[[torch.Tensor], torch.Tensor], x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    with torch.enable_grad():
        x = x.requires_grad_(True)
        g = grad_fn(x)
        dot = (g*v).sum()
        (hvp_x,) = torch.autograd.grad(dot, x, retain_graph=True)
    return hvp_x

def newton_cg_step(x: torch.Tensor, grad_fn, hvp_fn, damp: float, cg_iters: int):
    grad = grad_fn(x)
    def mv(p): return hvp_fn(x, p) + damp*p
    p = torch.zeros_like(x)
    r = -grad.clone(); z = r.clone(); rho = (r*z).sum()
    for _ in range(cg_iters):
        Ap = mv(z)
        alpha = rho / ((z*Ap).sum() + 1e-12)
        p = p + alpha*z
        r = r - alpha*Ap
        rho_new = (r*r).sum()
        if torch.sqrt(rho_new) < 1e-6: break
        z = r + (rho_new/(rho+1e-12))*z; rho = rho_new
    return p, grad

def run_xgd(cfg: CFG, y: torch.Tensor, k: torch.Tensor):
    x = upsample_nn(y, cfg.ds).detach()
    Atrue = lambda z: A_true(cfg, z, k)

    r0 = torch.sqrt(((Atrue(x)-y)**2).mean()).item()
    thresh = cfg.eps_y_rel * r0
    curve = [r0]
    for _ in range(cfg.max_iters):
        x_var = x.clone().requires_grad_(True)
        res = ((Atrue(x_var)-y)**2).mean()
        (grad,) = torch.autograd.grad(res, x_var)
        grad = grad.detach()

        step = 1.0; f0 = res.detach()
        while True:
            x_new = (x - step*grad).clamp(0,1)
            f1 = ((Atrue(x_new)-y)**2).mean().detach()
            if f1 <= f0 - cfg.ls_beta*step*(grad*grad).mean():
                x = x_new; break
            step *= cfg.ls_alpha
            if step < 1e-4: break
        r = torch.sqrt(((Atrue(x)-y)**2).mean()).item()
        curve.append(r)
        if r <= thresh: break
    return x, curve

def run_lm(cfg: CFG, y: torch.Tensor, k: torch.Tensor):
    x = upsample_nn(y, cfg.ds).detach()
    Atrue = lambda z: A_true(cfg, z, k)

    r0 = torch.sqrt(((Atrue(x)-y)**2).mean()).item()
    thresh = cfg.eps_y_rel * r0

    def loss(z): return 0.5*((Atrue(z)-y)**2).mean()
    def grad_fn(z):
        z = z.requires_grad_(True)
        val = loss(z)
        (gx,) = torch.autograd.grad(val, z, create_graph=True)
        return gx

    damp = cfg.lm_damp_init; curve = [r0]
    for _ in range(cfg.max_iters):
        p, grad = newton_cg_step(x, grad_fn, lambda z,v: hvp(grad_fn, z, v), damp, cfg.cg_iters)
        step = 1.0; f0 = loss(x).detach(); grad_dot_p = (grad*p).mean()
        while True:
            x_new = (x + step*p).clamp(0,1)
            f1 = loss(x_new).detach()
            if f1 <= f0 + cfg.ls_beta * step * grad_dot_p:
                x = x_new; damp = max(damp*0.7, 1e-4); break
            step *= cfg.ls_alpha; damp = min(damp*1.3, 1.0)
            if step < 1e-4: break
        r = torch.sqrt(((Atrue(x)-y)**2).mean()).item()
        curve.append(r)
        if r <= thresh: break
    return x, curve


In [6]:
def run_dnet(cfg, model, y, k, nominal):
    Atrue = lambda z: A_true(cfg, z, k)
    with torch.no_grad():
        model.set_nominal(nominal)
        traj = model.forward_steps(y, cfg)
        r0 = torch.sqrt(((Atrue(traj[0])-y)**2).mean()).item()
        thresh = cfg.eps_y_rel * r0
        curve = [r0]
        for xt in traj[1:]:
            r = torch.sqrt(((Atrue(xt)-y)**2).mean()).item()
            curve.append(r)
            if r <= thresh: break
        return traj[-1], curve

def evaluate(cfg, model, TE, device, dtype, nominal):
    X,Y,K = TE
    rmse, iters = {"LM":[], "X-GD":[], "DNet":[]}, {"LM":[], "X-GD":[], "DNet":[]}
    curves = {"LM":[], "X-GD":[], "DNet":[]}
    for i in range(X.shape[0]):
        x, y, k = X[i:i+1].to(device), Y[i:i+1].to(device), K[i:i+1].to(device)
        xl, cl = run_lm(cfg, y, k)
        xg, cg = run_xgd(cfg, y, k)
        xd, cd = run_dnet(cfg, model, y, k, nominal)
        rmse["LM"].append(float(torch.sqrt(F.mse_loss(xl,x)).item()))
        rmse["X-GD"].append(float(torch.sqrt(F.mse_loss(xg,x)).item()))
        rmse["DNet"].append(float(torch.sqrt(F.mse_loss(xd,x)).item()))
        iters["LM"].append(len(cl)-1); iters["X-GD"].append(len(cg)-1); iters["DNet"].append(len(cd)-1)
        curves["LM"].append(cl); curves["X-GD"].append(cg); curves["DNet"].append(cd)
    s = {"samples":len(X),
         "final_RMSE_mean":{k:np.mean(v) for k,v in rmse.items()},
         "iters_mean":{k:np.mean(v) for k,v in iters.items()}}
    print(json.dumps(s, indent=2))
    os.makedirs(cfg.out_dir, exist_ok=True)
    json.dump(s, open(f"{cfg.out_dir}/summary.json","w"), indent=2)
    with open(f"{cfg.out_dir}/mean_convergence.csv","w",newline="") as f:
        w=csv.writer(f); w.writerow(["iteration","LM","X-GD","DNet"])
        L=max(max(map(len,curves["LM"])),max(map(len,curves["X-GD"])),max(map(len,curves["DNet"])))
        def pad(c): return np.pad(c,(0,L-len(c)),constant_values=c[-1])
        for i in range(L):
            f.write(f"{i},{np.mean([pad(c)[i] for c in curves['LM']]):.6f},{np.mean([pad(c)[i] for c in curves['X-GD']]):.6f},{np.mean([pad(c)[i] for c in curves['DNet']]):.6f}\n")
    print("Saved files:\n - summary.json\n - mean_convergence.csv\nOutput dir:", cfg.out_dir)


In [7]:
def main():
    set_seed(CFG.seed)
    device = get_device(CFG.device); dtype = CFG.dtype
    print("device:", device)
    ds_path = os.path.join(CFG.out_dir, "dataset.pt")
    if os.path.exists(ds_path):
        data = torch.load(ds_path, map_location=device)
        TR,VA,TE = data["TR"],data["VA"],data["TE"]
        print("loaded dataset from", ds_path)
    else:
        TR,VA,TE = make_dataset(CFG, device, dtype)
        torch.save({"TR":TR,"VA":VA,"TE":TE}, ds_path)
        print("saved dataset to", ds_path)
    nominal = NominalForward(CFG, device, dtype)
    model = DNet(CFG.steps_unroll).to(device, dtype)
    t0=time.time(); train(CFG, model, TR, VA, device, dtype, nominal); t1=time.time()
    print(f"training took {t1-t0:.1f}s")
    evaluate(CFG, model, TE, device, dtype, nominal)

if __name__=="__main__":
    main()


device: cuda
saved dataset to outputs/dataset.pt
[ep 01/12] loss=0.01623
[ep 02/12] loss=0.00558
[ep 03/12] loss=0.00497
[ep 04/12] loss=0.00476
[ep 05/12] loss=0.00474
[ep 06/12] loss=0.00472
[ep 07/12] loss=0.00470
[ep 08/12] loss=0.00469
[ep 09/12] loss=0.00467
[ep 10/12] loss=0.00467
[ep 11/12] loss=0.00467
[ep 12/12] loss=0.00464
training took 189.3s
{
  "samples": 400,
  "final_RMSE_mean": {
    "LM": 0.08831002961844206,
    "X-GD": 0.12705526353791355,
    "DNet": 0.06374360848218202
  },
  "iters_mean": {
    "LM": 69.25,
    "X-GD": 80.0,
    "DNet": 6.0
  }
}
Saved files:
 - summary.json
 - mean_convergence.csv
Output dir: outputs
